In [2]:
!apt-get install -y glpk-utils
!pip install pyomo pandas matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
Suggested packages:
  libiodbc2-dev
The following NEW packages will be installed:
  glpk-utils libamd2 libcolamd2 libglpk40 libsuitesparseconfig5
0 upgraded, 5 newly installed, 0 to remove and 41 not upgraded.
Need to get 625 kB of archives.
After this operation, 2,158 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsuitesparseconfig5 amd64 1:5.10.1+dfsg-4build1 [10.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libamd2 amd64 1:5.10.1+dfsg-4build1 [21.6 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libcolamd2 amd64 1:5.10.1+dfsg-4build1 [18.0 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libglpk40 amd64 5.0-1 [361 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 glpk-ut

In [3]:
!glpsol --version

GLPSOL--GLPK LP/MIP Solver 5.0
Copyright (C) 2000-2020 Free Software Foundation, Inc.

This program has ABSOLUTELY NO WARRANTY.

This program is free software; you may re-distribute it under the terms
of the GNU General Public License version 3 or later.


In [4]:
# dessem_model.py
# Simplified DESSEM-like linear dispatch model using Pyomo.
# Outputs: ./dessem_outputs/results.csv and PNG plots (if matplotlib/pandas available).
#
# Requirements:
#   pip install pyomo pandas matplotlib
#   sudo apt-get install glpk-utils    # or use CBC/other solver

from pathlib import Path
import csv
import pyomo.environ as pyo

try:
    import pandas as pd
    import matplotlib.pyplot as plt
except Exception:
    pd = None
    plt = None

OUTPUT_DIR = Path("dessem_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_model(periods=3):
    model = pyo.ConcreteModel()
    model.T = pyo.RangeSet(1, periods)

    # Parameters (example values — adjust to your data)
    A = {1: 150.0, 2: 120.0, 3: 100.0}    # inflows (units consistent)
    D = {1: 180.0, 2: 200.0, 3: 190.0}    # demand (MW)
    C_term = 250.0
    C_def = 5000.0

    Ghid_min, Ghid_max = 0.0, 150.0
    Gterm_min, Gterm_max = 0.0, 200.0

    V0 = 300.0
    Vmin, Vmax = 200.0, 500.0

    # expose params on model for later use
    model.A = pyo.Param(model.T, initialize=A, mutable=True)
    model.D = pyo.Param(model.T, initialize=D, mutable=True)
    model.C_term = C_term
    model.C_def = C_def
    model.Ghid_min = Ghid_min
    model.Ghid_max = Ghid_max
    model.Gterm_min = Gterm_min
    model.Gterm_max = Gterm_max
    model.V0 = V0
    model.Vmin = Vmin
    model.Vmax = Vmax

    # Variables
    model.Ghid = pyo.Var(model.T, domain=pyo.NonNegativeReals)
    model.Gterm = pyo.Var(model.T, domain=pyo.NonNegativeReals)
    model.Def = pyo.Var(model.T, domain=pyo.NonNegativeReals)
    model.Q = pyo.Var(model.T, domain=pyo.NonNegativeReals)
    model.V = pyo.Var(model.T, domain=pyo.NonNegativeReals)

    # Objective
    def obj_rule(m):
        return sum(m.C_term * m.Gterm[t] + m.C_def * m.Def[t] for t in m.T)
    model.Obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # Constraints
    def balance_rule(m, t):
        if t == 1:
            return m.V[t] == m.V0 + m.A[t] - m.Q[t]
        else:
            return m.V[t] == m.V[t-1] + m.A[t] - m.Q[t]
    model.Balance = pyo.Constraint(model.T, rule=balance_rule)

    def volume_limits(m, t):
        return pyo.inequality(m.Vmin, m.V[t], m.Vmax)
    model.VolumeLimits = pyo.Constraint(model.T, rule=volume_limits)

    def ghid_limits(m, t):
        return pyo.inequality(m.Ghid_min, m.Ghid[t], m.Ghid_max)
    model.GhidLimits = pyo.Constraint(model.T, rule=ghid_limits)

    def gterm_limits(m, t):
        return pyo.inequality(m.Gterm_min, m.Gterm[t], m.Gterm_max)
    model.GtermLimits = pyo.Constraint(model.T, rule=gterm_limits)

    def demand_rule(m, t):
        return m.Ghid[t] + m.Gterm[t] + m.Def[t] == pyo.value(m.D[t])
    model.Demand = pyo.Constraint(model.T, rule=demand_rule)

    return model

def solve_model(model, solver_name="glpk"):
    solver = pyo.SolverFactory(solver_name)
    if not solver.available():
        raise RuntimeError(f"Solver '{solver_name}' not available.")
    result = solver.solve(model, tee=False)
    return result

def export_results(model, filename=OUTPUT_DIR/"results.csv"):
    rows = []
    total_cost = 0.0
    for t in model.T:
        ghid = pyo.value(model.Ghid[t])
        gterm = pyo.value(model.Gterm[t])
        q = pyo.value(model.Q[t])
        v = pyo.value(model.V[t])
        deficit = pyo.value(model.Def[t])
        cost = model.C_term * gterm + model.C_def * deficit
        total_cost += cost
        rows.append({
            "period": int(t),
            "Ghid": ghid,
            "Gterm": gterm,
            "Q": q,
            "Volume": v,
            "Deficit": deficit,
            "Cost": cost
        })
    keys = ["period", "Ghid", "Gterm", "Q", "Volume", "Deficit", "Cost"]
    with open(filename, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Results exported to {filename}")
    if pd:
        return pd.DataFrame(rows)
    return rows

def plot_results(df):
    if plt is None or df is None:
        print("matplotlib or pandas not available; skipping plots.")
        return
    # Generation plot
    fig1, ax1 = plt.subplots()
    ax1.plot(df["period"], df["Ghid"], marker="o")
    ax1.plot(df["period"], df["Gterm"], marker="o")
    ax1.set_xlabel("Period")
    ax1.set_ylabel("Generation (MW)")
    ax1.set_title("Geração Hidráulica e Térmica por Período")
    ax1.legend(["Ghid", "Gterm"])
    fig1.savefig(OUTPUT_DIR / "generation.png")
    plt.close(fig1)

    # Reservoir volume
    fig2, ax2 = plt.subplots()
    ax2.plot(df["period"], df["Volume"], marker="o")
    ax2.set_xlabel("Period")
    ax2.set_ylabel("Volume")
    ax2.set_title("Volume do Reservatório")
    fig2.savefig(OUTPUT_DIR / "volume.png")
    plt.close(fig2)
    print(f"Plots saved in {OUTPUT_DIR}")

def run_demo(solver="glpk"):
    model = build_model(periods=3)
    print("Model built. Solving...")
    solve_model(model, solver_name=solver)
    print("Solved. Exporting results...")
    df = export_results(model)
    plot_results(df)
    print("Done.")

if __name__ == "__main__":
    try:
        run_demo(solver="glpk")
    except Exception as e:
        print("Error:", e)
        print("Ensure Pyomo and a solver (glpk/cbc) are installed.")

Model built. Solving...
Solved. Exporting results...
Results exported to dessem_outputs/results.csv
Plots saved in dessem_outputs
Done.
